In [0]:
use catalog classic_workspace;
use schema default;

In [0]:
CREATE OR REPLACE TABLE orders (
  order_id    INT          COMMENT 'Unique order identifier',
  customer_id INT NOT NULL COMMENT 'Must reference a valid customer',
  amount      DOUBLE       COMMENT 'Order total in USD',
  status      STRING       COMMENT 'Order lifecycle status',
  created_at  TIMESTAMP    COMMENT 'Order creation timestamp'
)
USING DELTA
COMMENT 'Order management — constraints demo';

In [0]:
select * from orders;

In [0]:
INSERT INTO orders VALUES
  (1, 101, 149.99, 'pending',    '2024-08-01 10:00:00'),
  (2, 102, 320.00, 'completed',  '2024-08-01 10:05:00'),
  (3, 103, 89.50,  'processing', '2024-08-01 10:10:00');

In [0]:
select * from orders;

In [0]:
INSERT INTO orders VALUES
  (4, 104, 55.00, 'pending', '2024-08-01 11:00:00');

In [0]:
select * from orders

In [0]:
ALTER TABLE orders
ADD CONSTRAINT valid_amount CHECK (amount > 0);

In [0]:
INSERT INTO orders VALUES
  (5, 105, 50.00, 'pending', '2024-08-01 11:05:00');

In [0]:
select * from orders;

In [0]:
ALTER TABLE orders
ADD CONSTRAINT valid_status
CHECK (status IN ('pending', 'processing', 'completed', 'cancelled'));

In [0]:
show table properties orders;

In [0]:
INSERT INTO orders VALUES
  (7, 106, 210.00, 'shipped', '2024-08-01 11:10:00');

In [0]:
select * from orders;

In [0]:
ALTER TABLE orders DROP CONSTRAINT valid_status;

In [0]:
select * from orders;

In [0]:
show tblproperties classic_workspace.default.orders;

In [0]:
CREATE OR REPLACE TABLE customers_dim (
  customer_sk  BIGINT GENERATED ALWAYS AS IDENTITY
               COMMENT 'Surrogate key — Delta-managed, always auto-generated',
  customer_id  STRING  NOT NULL COMMENT 'Business key from source system',
  name         STRING  NOT NULL,
  email        STRING,
  tier         STRING,
  is_current   BOOLEAN DEFAULT true
)
USING DELTA
COMMENT 'Customer dimension — star schema surrogate key demo'
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
INSERT INTO customers_dim (customer_id, name, email, tier, is_current) VALUES
  ('C001', 'Alice Nguyen',  'alice@example.com',  'gold',     true),
  ('C002', 'Bob Patel',     'bob@example.com',    'silver',   true),
  ('C003', 'Carol Santos',  'carol@example.com',  'platinum', true);

In [0]:
SELECT customer_sk, customer_id, name, tier
FROM   customers_dim
ORDER  BY customer_sk;

In [0]:
INSERT INTO customers_dim
  (customer_sk, customer_id, name, email, tier, is_current)
VALUES
  (99, 'C004', 'David Kim', 'david@example.com', 'bronze', true);

In [0]:
CREATE OR REPLACE TABLE customers_dim_v2 (
  customer_sk  BIGINT GENERATED BY DEFAULT AS IDENTITY,
  customer_id  STRING NOT NULL,
  name         STRING NOT NULL,
  email        STRING,
  tier         STRING
)
USING DELTA
COMMENT 'Customer dim v2 — BY DEFAULT for migration scenarios';

In [0]:
INSERT INTO customers_dim_v2
  (customer_sk, customer_id, name, email, tier)
VALUES
  (5001, 'C001', 'Alice Nguyen',  'alice@example.com',  'gold'),
  (5002, 'C002', 'Bob Patel',     'bob@example.com',    'silver');

In [0]:
INSERT INTO customers_dim_v2 (customer_id, name, email, tier)
VALUES ('C003', 'Carol Santos', 'carol@example.com', 'platinum');

In [0]:
SELECT customer_sk, customer_id, name
FROM   customers_dim_v2
ORDER  BY customer_sk;